# WSJ quarterly sentiment panel for the Expectations project (in-TDM feature reduction)

Reads the combined full-text parquet built by `proquest_tdm_fulltext_combine.ipynb`
(`wsj_full_corpus.parquet`: `date, headline, body, article_id`), scores every article
with three lexicons, and reduces the corpus to a **compact quarterly feature panel** that
fits the 30 MB export cap and ships to the DCC model alongside the macro-news features.

**Model targets (drive the custom feature design):** discount-rate expectations,
earnings-growth expectations, dividend-growth expectations. The custom list below carries a
salience channel aligned to each (`dr_salience`, `earn_salience`, `div_salience`) plus a
general directional tone (`cf_growth`/`cf_decline`).

**Lexicons (all count-verified against published numbers):**
- **Loughran–McDonald** master dictionary — Negative / Positive / Uncertainty / Litigious /
  Strong-Modal / Weak-Modal / Constraining. Finance-standard, identical instrument 1889→2026.
- **Harvard General Inquirer "Negativ"** — Tetlock (2007) pessimism factor (+ Positiv).
- **Custom cash-flow / discount-rate list** — author-curated, *inline below* (editable).

**Design choices (locked with Rafael):**
- Quarterly frequency, market-level aggregate (matches the expectations series).
- Scores on **body and headline separately** (body leads — incremental to macro news;
  headlines collapse into the macro state).
- Per quarter: **token-weighted level** for each category *plus* **cross-article dispersion**
  (disagreement — orthogonal to macro by construction), with `n_articles` and `ocr_quality`.
- Pure-pyarrow streaming read, no pandas↔arrow bridge (TDM pyarrow chokes on it).

**Files to upload into TDM with this notebook (same folder):**
`LM_MasterDictionary.csv`, `inqtabs.txt`.

**Output:** `output_files/wsj_sentiment_quarterly.{csv,parquet}` + `figures/wsj_sentiment_overview.png`.


## Cell 1 — load & verify lexicons (run first)

Loads the two uploaded dictionary files, rebuilds the category word-sets, defines the
inline custom lists, and **prints counts so you can confirm the right files loaded**
(LM Negative 2355, Positive 354, Uncertainty 297, … ; GI Negativ 2291 entries → 2005 words).

In [ ]:
import re, csv, os, time, math
from collections import defaultdict
from pathlib import Path
import pyarrow.parquet as pq

# --- where the uploaded dictionaries live (this notebook's folder, with fallbacks) ---
def find_file(name, extra=()):
    for base in (Path('.'), Path('lexicons'), Path('..'),
                 Path('/home/jovyan'), *map(Path, extra)):
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError(f'{name} not found — upload it next to this notebook.')

LM_PATH = find_file('LM_MasterDictionary.csv')
GI_PATH = find_file('inqtabs.txt')
print(f'LM : {LM_PATH}')
print(f'GI : {GI_PATH}\n')

# --- Loughran-McDonald: word in a category if its year-column is non-zero ---
LM_CATS = ['Negative','Positive','Uncertainty','Litigious',
           'Strong_Modal','Weak_Modal','Constraining']
lm_sets = {c: set() for c in LM_CATS}
LM_VOCAB = set()                       # full master word list -> English reference for OCR
with open(LM_PATH, encoding='latin-1') as fh:
    for r in csv.DictReader(fh):
        w = r['Word'].strip().upper()
        if not w:
            continue
        LM_VOCAB.add(w)
        for c in LM_CATS:
            v = r[c].strip()
            if v and v != '0':
                lm_sets[c].add(w)

# --- Harvard GI: Tetlock uses the "Negativ" column (also keep "Positiv") ---
gi_neg, gi_pos = set(), set()
with open(GI_PATH, encoding='latin-1') as fh:
    for r in csv.DictReader(fh, delimiter='\t'):
        base = r['Entry'].split('#')[0].strip().upper()    # drop sense suffix WORD#1
        if not base:
            continue
        if r.get('Negativ','').strip():
            gi_neg.add(base)
        if r.get('Positiv','').strip():
            gi_pos.add(base)

# --- Custom lists (AUTHOR-CURATED, not canonical — edit freely) -------------------
# One salience channel per model target, plus a general directional tone.
# Salience = "how much is this quarter ABOUT X" (direction-neutral);
# cf_growth/cf_decline = expansion vs contraction (general direction).
# Overlap with LM/GI is intentional (different, target-aligned lens); the DCC redundancy
# filter prunes anything that doesn't add over the macro panel.

# earnings-growth target
earn_salience = {
    'EARNINGS','EARNING','PROFIT','PROFITS','PROFITABILITY','REVENUE','REVENUES','MARGIN',
    'MARGINS','SALES','INCOME','OUTPUT','ORDERS','BACKLOG','PRODUCTION','SHIPMENTS',
    'BOOKINGS','TURNOVER','GUIDANCE','OUTLOOK','RECEIPTS',
}
# dividend-growth target
div_salience = {
    # core (unambiguous dividend / distribution vocabulary)
    'DIVIDEND','DIVIDENDS','PAYOUT','PAYOUTS','DISTRIBUTION','DISTRIBUTIONS',
    'DISBURSEMENT','DISBURSEMENTS','DISBURSE','DISBURSED','SCRIP','PAYABLE',
    # dividend events / actions (financial context; may catch some non-dividend uses)
    'DECLARED','DECLARE','DECLARES','DECLARATION','OMITTED','OMIT','OMITS','OMISSION',
    'SUSPENDED','SUSPENSION','RESUMED','RESUMPTION','ARREARS','INTERIM','PREFERRED',
}
# discount-rate / required-return / risk target
dr_salience = {
    'DISCOUNT','DISCOUNTING','RISK','RISKS','RISKY','PREMIUM','PREMIUMS','PREMIA','YIELD',
    'YIELDS','INTEREST','RATES','VOLATILITY','VOLATILE','CREDIT','DEFAULT','DEFAULTS',
    'SPREAD','SPREADS','BOND','BONDS','TREASURY','TREASURIES','INFLATION','LIQUIDITY',
    'MONETARY','TIGHTENING','TIGHTEN','EASING','REQUIRED','RETURNS',
}
# general directional tone (applies to all growth targets)
cf_growth = {
    'GROWTH','GROW','GROWS','GROWING','GREW','EXPAND','EXPANDS','EXPANDING','EXPANDED',
    'EXPANSION','RISE','RISES','RISING','ROSE','RISEN','INCREASE','INCREASES','INCREASING',
    'INCREASED','GAIN','GAINS','GAINING','GAINED','SURGE','SURGED','SURGES','SURGING',
    'BOOM','BOOMING','STRONG','STRENGTH','STRONGER','ROBUST','IMPROVE','IMPROVES',
    'IMPROVING','IMPROVED','IMPROVEMENT','RECOVERY','RECOVER','RECOVERED','RECOVERING',
    'ACCELERATE','ACCELERATED','ACCELERATING','UPTURN','ADVANCE','ADVANCED','ADVANCES',
    'ADVANCING','PROSPER','PROSPEROUS','PROSPERITY','EXCEED','EXCEEDED','EXCEEDS','REBOUND',
}
cf_decline = {
    'DECLINE','DECLINES','DECLINING','DECLINED','FALL','FALLS','FALLING','FELL','FALLEN',
    'DROP','DROPS','DROPPING','DROPPED','DECREASE','DECREASES','DECREASING','DECREASED',
    'CONTRACT','CONTRACTION','CONTRACTED','CONTRACTING','SHRINK','SHRINKS','SHRINKING',
    'SHRANK','SHRUNK','SLUMP','SLUMPED','SLUMPING','RECESSION','DEPRESSION','LOSS','LOSSES',
    'WEAK','WEAKNESS','WEAKER','WEAKEN','WEAKENED','WEAKENING','SLOWDOWN','SLOWING','SLOWED',
    'DOWNTURN','PLUNGE','PLUNGED','PLUNGES','PLUNGING','DETERIORATE','DETERIORATED',
    'DETERIORATING','DETERIORATION','COLLAPSE','COLLAPSED','REDUCTION',
}

# Master mapping name -> word-set. Everything downstream iterates this.
LEXICONS = {
    'lm_neg': lm_sets['Negative'],   'lm_pos': lm_sets['Positive'],
    'lm_unc': lm_sets['Uncertainty'],'lm_lit': lm_sets['Litigious'],
    'lm_strong': lm_sets['Strong_Modal'], 'lm_weak': lm_sets['Weak_Modal'],
    'lm_constr': lm_sets['Constraining'],
    'gi_neg': gi_neg, 'gi_pos': gi_pos,
    'earn_salience': earn_salience, 'div_salience': div_salience, 'dr_salience': dr_salience,
    'cf_growth': cf_growth, 'cf_decline': cf_decline,
}

print('Loaded lexicon counts (verify against published):')
print(f'  LM vocab (OCR reference): {len(LM_VOCAB):,}')
for k in ['lm_neg','lm_pos','lm_unc','lm_lit','lm_strong','lm_weak','lm_constr']:
    print(f'  {k:13} {len(LEXICONS[k]):5}')
print(f'  gi_neg        {len(gi_neg):5}   (2291 GI Negativ entries -> unique word strings)')
print(f'  gi_pos        {len(gi_pos):5}   (1915 GI Positiv entries -> unique word strings)')
for k in ['earn_salience','div_salience','dr_salience','cf_growth','cf_decline']:
    print(f'  {k:13} {len(LEXICONS[k]):5}   (custom)')

# Combined-corpus parquet from the previous notebook (resolved lazily in Cells 3-4).
def resolve_parquet():
    cands = [
        Path('../ProQuest TDM Studio Samples/output_files/wsj_full_corpus.parquet'),
        Path('../../ProQuest TDM Studio Samples/output_files/wsj_full_corpus.parquet'),
        Path('output_files/wsj_full_corpus.parquet'),
        Path('../output_files/wsj_full_corpus.parquet'),
    ]
    p = next((c for c in cands if c.exists()), None)
    if p is None:
        raise FileNotFoundError('wsj_full_corpus.parquet not found — set the path manually.')
    return p


## Cell 2 — scoring functions (run second)

`score(text)` tokenizes (letters only, uppercased), then uses `Counter` ∩ word-set
intersections (fast — a few hundred unique tokens vs. the full lexicons) to return raw
category counts, the token total, and the in-vocab count for OCR quality.

In [ ]:
from collections import Counter

TOKEN_RE = re.compile(r"[A-Za-z]+")

def score(text):
    """Return (n_tokens, counts_dict, in_vocab) for one document."""
    toks = TOKEN_RE.findall(text or '')
    n = len(toks)
    if n == 0:
        return 0, {k: 0 for k in LEXICONS}, 0
    c = Counter(t.upper() for t in toks)
    vocab = set(c)
    counts = {name: sum(c[w] for w in (vocab & s)) for name, s in LEXICONS.items()}
    in_vocab = sum(c[w] for w in (vocab & LM_VOCAB))
    return n, counts, in_vocab

# Per-article derived fractions we track dispersion (disagreement) on — body only.
# tone = positive minus negative, as a fraction of tokens.
def article_fracs(n, counts):
    if n == 0:
        return {}
    f = {k: counts[k] / n for k in counts}
    f['lm_tone'] = (counts['lm_pos'] - counts['lm_neg']) / n
    f['cf_tone'] = (counts['cf_growth'] - counts['cf_decline']) / n
    return f

DISP_FEATURES = ['lm_tone', 'lm_neg', 'lm_unc', 'gi_neg', 'cf_tone']
print('Scoring ready. Dispersion tracked for:', DISP_FEATURES)


## Cell 3 — peek: score a few real articles before the full run

Reads the first row group of the combined parquet and scores 3 articles, printing token
counts, key category fractions, and OCR quality. **Confirm body fractions are non-zero and
OCR quality looks sane (modern eras ≳0.9; early OCR lower) before launching Cell 4.**

In [ ]:
PARQUET = resolve_parquet()
print(f'Combined corpus: {PARQUET}  ({os.path.getsize(PARQUET)/1e6:,.1f} MB)\n')

pf = pq.ParquetFile(PARQUET)
rg = pf.read_row_group(0)
cols = {c: rg.column(c).to_pylist() for c in ['date','headline','body','article_id']}

for i in range(min(3, rg.num_rows)):
    body, head = cols['body'][i] or '', cols['headline'][i] or ''
    nb, cb, vb = score(body)
    nh, ch, vh = score(head)
    fb = article_fracs(nb, cb)
    print('=' * 78)
    print(f"[{cols['date'][i]}] id={cols['article_id'][i]}  |  {head[:70]}")
    print(f'  body tokens={nb:,}  headline tokens={nh}')
    if nb:
        print(f'  body  lm_neg={fb["lm_neg"]:.4f}  lm_pos={fb["lm_pos"]:.4f}  '
              f'lm_unc={fb["lm_unc"]:.4f}  gi_neg={fb["gi_neg"]:.4f}')
        print(f'        lm_tone={fb["lm_tone"]:+.4f}  cf_tone={fb["cf_tone"]:+.4f}')
        print(f'        earn={fb["earn_salience"]:.4f}  div={fb["div_salience"]:.4f}  '
              f'dr={fb["dr_salience"]:.4f}')
        print(f'        OCR quality (in-vocab share)={vb/nb:.3f}')
    print(f'  body[:200]: {body[:200]!r}')


## Cell 4 — full streaming pass → quarterly panel

Streams the corpus row-group by row-group, scores body + headline per article, and
accumulates into per-quarter running totals (only ~550 quarters held in memory — the text
is never all in RAM). Produces token-weighted levels, cross-article dispersion, and
coverage columns. Expect this to run a while (millions of full-text articles, CPU).

In [ ]:
PARQUET = resolve_parquet()
pf = pq.ParquetFile(PARQUET)

def new_q():
    return {
        'n_art': 0, 'sum_tok': 0, 'sum_invocab': 0,
        'hits': defaultdict(int),          # body token-weighted level
        'hl_sum_tok': 0, 'hl_hits': defaultdict(int),
        'fsum': defaultdict(float), 'fsq': defaultdict(float),   # body dispersion
    }
agg = defaultdict(new_q)
n_skip_nodate = 0
t0 = time.time()
done = 0

for rgi in range(pf.num_row_groups):
    rg = pf.read_row_group(rgi)
    dates = rg.column('date').to_pylist()
    bodies = rg.column('body').to_pylist()
    heads = rg.column('headline').to_pylist()
    for d, body, head in zip(dates, bodies, heads):
        if d is None:
            n_skip_nodate += 1
            continue
        q = (d.year, (d.month - 1) // 3 + 1)
        a = agg[q]
        nb, cb, vb = score(body)
        nh, ch, vh = score(head)
        a['n_art'] += 1
        a['sum_tok'] += nb
        a['sum_invocab'] += vb
        for k, v in cb.items():
            a['hits'][k] += v
        a['hl_sum_tok'] += nh
        for k, v in ch.items():
            a['hl_hits'][k] += v
        if nb:
            fb = article_fracs(nb, cb)
            for k in DISP_FEATURES:
                a['fsum'][k] += fb[k]
                a['fsq'][k] += fb[k] * fb[k]
    done += rg.num_rows
    rate = done / (time.time() - t0)
    print(f'  row group {rgi+1}/{pf.num_row_groups}  ({done:,} articles, {rate:.0f}/s)',
          flush=True)

print(f'\nQuarters: {len(agg):,} | articles scored: {done - n_skip_nodate:,} | '
      f'skipped (no date): {n_skip_nodate:,}')


## Cell 5 — assemble the panel and write it (CSV + parquet)

Token-weighted levels (`hits / tokens`), derived tone, headline columns (`hl_`),
body dispersion (`*_disp`), and coverage (`n_articles`, `mean_body_tokens`, `ocr_quality`).
Written with pandas `to_csv` (no arrow bridge) plus a zstd parquet via explicit pyarrow.

In [ ]:
import pandas as pd
import pyarrow as pa
from datetime import date as _date

LEVEL_CATS = list(LEXICONS.keys())

rows = []
for (y, q) in sorted(agg.keys()):
    a = agg[(y, q)]
    st = a['sum_tok'] or 1
    hst = a['hl_sum_tok'] or 1
    na = a['n_art'] or 1
    row = {'quarter': f'{y}Q{q}', 'quarter_start': _date(y, 3 * (q - 1) + 1, 1),
           'year': y, 'q': q, 'n_articles': a['n_art']}
    for k in LEVEL_CATS:
        row[k] = a['hits'][k] / st
    row['lm_tone'] = (a['hits']['lm_pos'] - a['hits']['lm_neg']) / st
    row['cf_tone'] = (a['hits']['cf_growth'] - a['hits']['cf_decline']) / st
    for k in LEVEL_CATS:
        row['hl_' + k] = a['hl_hits'][k] / hst
    row['hl_lm_tone'] = (a['hl_hits']['lm_pos'] - a['hl_hits']['lm_neg']) / hst
    row['hl_cf_tone'] = (a['hl_hits']['cf_growth'] - a['hl_hits']['cf_decline']) / hst
    for k in DISP_FEATURES:
        m = a['fsum'][k] / na
        var = max(0.0, a['fsq'][k] / na - m * m)
        row[k + '_disp'] = math.sqrt(var)
    row['mean_body_tokens'] = a['sum_tok'] / na
    row['ocr_quality'] = a['sum_invocab'] / st
    rows.append(row)

panel = pd.DataFrame(rows).sort_values(['year', 'q']).reset_index(drop=True)

# --- Aggregate sentiment index (single interpretable series) --------------------
# Equal-weighted composite of the three polarity tones (LM, GI/Tetlock, cash-flow).
# These tones live on different scales (lexicon sizes differ a lot), so we put them
# on a common footing with a single FULL-SAMPLE z-score, then average. This is a
# fixed cross-quarter rescaling to balance the three lexicons — NOT a temporal /
# rolling transform. Each quarter's value is just that quarter's news sentiment;
# the model handles the time dimension and can re-standardize on its train window.
# Oriented so higher = more optimistic.
_tones = pd.DataFrame({
    'lm_tone': panel['lm_tone'],
    'gi_tone': panel['gi_pos'] - panel['gi_neg'],
    'cf_tone': panel['cf_tone'],
})
_z = (_tones - _tones.mean()) / _tones.std()
panel['sent_agg'] = _z.mean(axis=1)

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUT_DIR / 'wsj_sentiment_quarterly.csv'
PQ_PATH  = OUT_DIR / 'wsj_sentiment_quarterly.parquet'

panel.to_csv(CSV_PATH, index=False)   # pure pandas, no arrow bridge

arrays, names = [], []
for col in panel.columns:
    s = panel[col]
    if col == 'quarter_start':
        arr = pa.array([pa.scalar(v).as_py() if v is not None else None for v in s],
                       type=pa.date32())
    elif s.dtype == object:
        arr = pa.array(s.astype(str).tolist(), type=pa.string())
    elif str(s.dtype).startswith('int'):
        arr = pa.array(s.tolist(), type=pa.int64())
    else:
        arr = pa.array(s.tolist(), type=pa.float64())
    arrays.append(arr); names.append(col)
pq.write_table(pa.Table.from_arrays(arrays, names=names), PQ_PATH,
               compression='zstd', compression_level=19)

print(f'Wrote {CSV_PATH}  ({os.path.getsize(CSV_PATH)/1e6:.2f} MB)')
print(f'Wrote {PQ_PATH}   ({os.path.getsize(PQ_PATH)/1e6:.2f} MB)')
print(f'Panel: {panel.shape[0]} quarters x {panel.shape[1]} columns')
print(f'Span : {panel["quarter"].iloc[0]} -> {panel["quarter"].iloc[-1]}')
print('\nColumns:', list(panel.columns))


## Cell 6 — peek at the panel + coverage sanity

First/last quarters, coverage over time, and any all-NaN columns. Watch `n_articles` and
`ocr_quality` in the early decades — thin or low-OCR quarters are where the series is least
reliable (carry these into the DCC model so it can down-weight them).

In [ ]:
import pandas as pd
from pathlib import Path
panel = pd.read_csv(Path('../ProQuest TDM Studio Samples/output_files/wsj_sentiment_quarterly.csv'))

show = ['quarter','n_articles','ocr_quality','sent_agg',
        'lm_tone','lm_unc','gi_neg','cf_tone','earn_salience','div_salience','dr_salience']
print('First 4 quarters:')
print(panel[show].head(4).to_string(index=False))
print('\nLast 4 quarters:')
print(panel[show].tail(4).to_string(index=False))

print('\nCoverage by decade (article counts):')
panel['decade'] = (panel['year'] // 10) * 10
print(panel.groupby('decade')['n_articles'].sum().to_string())
print('\nOCR quality by decade (mean):')
print(panel.groupby('decade')['ocr_quality'].mean().round(3).to_string())

nan_cols = [c for c in panel.columns if panel[c].isna().any()]
print('\nColumns with any NaN:', nan_cols if nan_cols else 'none')


## Cell 7 — plot the sentiment series for inspection

Five stacked panels over the full span (raw quarterly thin, 4-quarter rolling mean thick):
**(a)** the aggregate sentiment index (z-scored composite, + = optimistic),
**(b)** overall tone & Tetlock pessimism, **(c)** uncertainty & discount-rate salience,
**(d)** earnings vs. dividend salience & directional tone, **(e)** coverage (articles, log)
& OCR quality. Reference lines at 1929 / 1973 / 2008 / 2020. Saves a PNG and shows inline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
panel = pd.read_csv(OUT_DIR / 'wsj_sentiment_quarterly.csv', parse_dates=['quarter_start'])
x = panel['quarter_start']
def roll(col):
    return panel[col].rolling(4, min_periods=1, center=True).mean()

FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
refs = [pd.Timestamp('1929-09-01'), pd.Timestamp('1973-10-01'),
        pd.Timestamp('2008-09-01'), pd.Timestamp('2020-03-01')]

fig, ax = plt.subplots(5, 1, figsize=(11, 15), sharex=True)

def add_refs(a):
    for r in refs:
        a.add_artist(plt.Line2D([r, r], a.get_ylim(), color='0.7', lw=0.8, ls='--', zorder=0))

def dual(a, left_col, left_lab, right_col, right_lab, lc='C0', rc='C3'):
    a.plot(x, panel[left_col], lw=0.5, alpha=0.35, color=lc)
    a.plot(x, roll(left_col), lw=1.7, color=lc, label=left_lab)
    a.set_ylabel(left_lab, color=lc); a.tick_params(axis='y', labelcolor=lc)
    b = a.twinx()
    b.plot(x, panel[right_col], lw=0.5, alpha=0.35, color=rc)
    b.plot(x, roll(right_col), lw=1.7, color=rc, label=right_lab)
    b.set_ylabel(right_lab, color=rc); b.tick_params(axis='y', labelcolor=rc)
    return b

# (a) AGGREGATE sentiment index (z-scored composite, + = optimistic)
ax[0].plot(x, panel['sent_agg'], lw=0.5, alpha=0.35, color='C0')
ax[0].plot(x, roll('sent_agg'), lw=1.9, color='C0')
ax[0].axhline(0, color='0.6', lw=0.8)
ax[0].set_ylabel('aggregate sentiment\n(z-scored, + = optimistic)')
ax[0].set_title('(a) Aggregate sentiment index', loc='left', fontsize=10)
# (b) tone & Tetlock pessimism
dual(ax[1], 'lm_tone', 'LM tone (pos-neg)', 'gi_neg', 'GI Negativ (Tetlock)')
ax[1].set_title('(b) Overall tone and Tetlock pessimism', loc='left', fontsize=10)
# (c) uncertainty & discount-rate salience
dual(ax[2], 'lm_unc', 'LM uncertainty', 'dr_salience', 'Discount-rate salience')
ax[2].set_title('(c) Uncertainty and discount-rate salience', loc='left', fontsize=10)
# (d) earnings vs dividend salience + directional tone
ax[3].plot(x, roll('earn_salience'), lw=1.7, color='C0', label='earnings salience')
ax[3].plot(x, roll('div_salience'),  lw=1.7, color='C2', label='dividend salience')
ax[3].set_ylabel('salience'); axd3 = ax[3].twinx()
axd3.plot(x, roll('cf_tone'), lw=1.5, color='C1', label='cf tone (growth-decline)')
axd3.set_ylabel('cf tone', color='C1'); axd3.tick_params(axis='y', labelcolor='C1')
ax[3].set_title('(d) Earnings vs dividend salience, directional tone', loc='left', fontsize=10)
ax[3].legend(loc='upper left', fontsize=8); axd3.legend(loc='upper right', fontsize=8)
# (e) coverage & OCR quality
ax[4].plot(x, panel['n_articles'], lw=1.0, color='C4', label='articles / quarter')
ax[4].set_yscale('log'); ax[4].set_ylabel('articles (log)', color='C4')
ax[4].tick_params(axis='y', labelcolor='C4')
axe = ax[4].twinx(); axe.plot(x, panel['ocr_quality'], lw=1.4, color='C5', label='OCR quality')
axe.set_ylabel('OCR quality', color='C5'); axe.set_ylim(0, 1)
axe.tick_params(axis='y', labelcolor='C5')
ax[4].set_title('(e) Coverage and OCR quality', loc='left', fontsize=10)
ax[4].set_xlabel('quarter')

for a in ax:
    add_refs(a)
fig.tight_layout()
PNG = FIG_DIR / 'wsj_sentiment_overview.png'
fig.savefig(PNG, dpi=140, bbox_inches='tight')
print(f'Saved {PNG}  ({os.path.getsize(PNG)/1e3:.0f} KB)')
plt.show()


## Cell 8 — diagnostics: are these measures signal or artifact?

Four checks on the panel: **(1)** correlation of every feature with `ocr_quality`, log article
count, and time — flags features that track data quality/era rather than economics;
**(2)** a reliable, well-covered subsample for re-checking; **(3)** correlation among the
headline features (redundancy); **(4)** a drop-in validation against your expectation
targets — put `expectations_targets.csv` (columns: `quarter`, then your series) in
`output_files/` and this reports which features actually track discount-rate / earnings /
dividend expectations where they exist (post-1981). That last one is the real test of what
these measures *say*.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
panel = pd.read_csv(OUT_DIR / 'wsj_sentiment_quarterly.csv', parse_dates=['quarter_start'])

# economic-content columns (exclude identifiers + the data-quality columns themselves)
EXCLUDE = {'quarter','quarter_start','year','q','decade',
           'n_articles','mean_body_tokens','ocr_quality'}
FEATURES = [c for c in panel.columns if c not in EXCLUDE]

# ---- 1) Artifact check: do features track DATA QUALITY / TIME rather than economics? ----
qual = pd.DataFrame({
    'corr_ocr_quality': [panel[f].corr(panel['ocr_quality']) for f in FEATURES],
    'corr_log_nartic' : [panel[f].corr(np.log(panel['n_articles'].clip(lower=1))) for f in FEATURES],
    'corr_time'       : [panel[f].corr(panel['year']) for f in FEATURES],
}, index=FEATURES).round(3)
qual['artifact_risk'] = (qual['corr_ocr_quality'].abs() > 0.5) | (qual['corr_time'].abs() > 0.6)
print('=== 1) Artifact check — corr with OCR quality / log articles / time ===')
print(qual.sort_values('corr_ocr_quality', key=lambda s: s.abs(), ascending=False).to_string())
flagged = qual.index[qual['artifact_risk']].tolist()
print('\nArtifact-risk (|corr ocr|>0.5 or |corr time|>0.6):', flagged or 'none')
print('-> for flagged features, residualize on ocr_quality before using, or use the '
      'reliable subsample below.')

# ---- 2) Reliable subsample (well-covered, decent-OCR quarters) ----
thr_n = panel['n_articles'].median()
rel = panel[(panel['n_articles'] >= thr_n) & (panel['ocr_quality'] >= 0.70)]
print(f'\n=== 2) Reliable subsample: n_articles>={thr_n:.0f} & ocr>=0.70 '
      f'({len(rel)}/{len(panel)} quarters, {rel["quarter"].iloc[0]}->{rel["quarter"].iloc[-1]}) ===')

# ---- 3) Headline-feature redundancy ----
MAIN = [c for c in ['sent_agg','lm_tone','gi_neg','lm_unc','lm_lit',
                    'earn_salience','div_salience','dr_salience','cf_tone',
                    'lm_tone_disp','gi_neg_disp'] if c in panel.columns]
print('\n=== 3) Correlation among headline features ===')
print(panel[MAIN].corr().round(2).to_string())

# ---- 4) Validation vs expectation targets (drop-in when available) ----
# Provide a CSV with a 'quarter' column ('1985Q1' style) + your target series, e.g.
# dr_exp / earn_exp / div_exp (discount-rate / earnings-growth / dividend-growth
# expectations). This reports which features actually track each target where it exists.
TARGETS_PATH = OUT_DIR / 'expectations_targets.csv'
print('\n=== 4) Feature vs expectation targets ===')
if TARGETS_PATH.exists():
    tg = pd.read_csv(TARGETS_PATH)
    target_cols = [c for c in tg.columns if c != 'quarter']
    m = panel.merge(tg, on='quarter', how='inner')
    print(f'matched {len(m)} quarters; targets: {target_cols}')
    corr = pd.DataFrame({t: [m[f].corr(m[t]) for f in FEATURES] for t in target_cols},
                        index=FEATURES).round(3)
    print(corr.to_string())
    for t in target_cols:
        s = corr[t].abs().sort_values(ascending=False)
        print(f'\nTop features for {t}: ' +
              ', '.join(f'{i} ({corr.loc[i, t]:+.2f})' for i in s.index[:6]))
else:
    print(f'No targets file at "{TARGETS_PATH.name}" — skipping.')
    print('Drop one in (columns: quarter, <your target series>) and re-run this cell.')


## Cell 9 — (optional) export the panel to the results bucket

Unlike the full-text corpus, the **panel (and the PNG) are small and meant to leave TDM**
for the DCC model. Uncomment to ship them (well under the 30 MB cap).

In [ ]:
# B = 's3://pq-tdm-studio-results/tdm-ale-data/a4992/results/'
# !aws s3 cp "../ProQuest TDM Studio Samples/output_files/wsj_sentiment_quarterly.csv" "$B"
# !aws s3 cp "../ProQuest TDM Studio Samples/output_files/figures/wsj_sentiment_overview.png" "$B"
print('Export cell — uncomment to ship the quarterly panel + overview PNG to DCC.')
